In [ ]:
from snowflake.snowpark.context import get_active_session
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

session = get_active_session()
data = session.table("APPROVAL_DB.GOLD.APPROVAL_OBT").to_pandas()

## Prepare the data

In [ ]:
approval = data.sort_values('DATE_DAY').reset_index(drop=True)

approval['TARGET_APPROVAL']     = approval['APPROVAL_RAW'].shift(-1)
approval['TARGET_DISAPPROVAL']  = approval['DISAPPROVAL_RAW'].shift(-1)

# Drop the last row, since we just shifted
approval = approval.dropna(subset=['TARGET_APPROVAL', 'TARGET_DISAPPROVAL'])

two_weeks_ago = datetime.now() - timedelta(weeks=2)
cutoff = two_weeks_ago.date()

train = approval[approval['DATE_DAY'] < cutoff]
test  = approval[approval['DATE_DAY'] >= cutoff]

DROP_COLS = ['DATE_DAY', 'TARGET_APPROVAL', 'TARGET_DISAPPROVAL'] # Add more to this. Current approach has data leakage

X_train = train.drop(columns=DROP_COLS)
y_train = train['TARGET_APPROVAL']
X_test  = test.drop(columns=DROP_COLS)
y_test  = test['TARGET_APPROVAL']

# Model training

In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(n_estimators = 300, max_depth = 4, learning_rate = 0.05, subsample= 0.8, random_state = 42)
model.fit(X_train, y_train)

preds = model.predict(X_test)

importance = pd.Series(model.feature_importances_, index=X_train.columns)
importance.sort_values(ascending=False).head(15)



In [ ]:
import matplotlib.pyplot as plt
x_axis = range(len(y_test))

plt.figure(figsize=(12,6))
# Actual values
plt.plot(x_axis, y_test, label="Actual (Test)", linewidth=2)

# Predictions
plt.plot(x_axis, preds, label="Predicted (XGBoost)", linestyle="--")

plt.title("Approval Rating: Actual vs Predicted")
plt.xlabel("Time / Index")
plt.ylabel("Approval")
plt.legend()

plt.show()